# LABORATORIJSKA VJEŽBA 8: Evolucione strategije (1+1)

**Predmet:** Optimizacija resursa  
**Univerzitet u Sarajevu - Elektrotehnički fakultet**

---

## Uvod

Algoritam ES(1+1) je najjednostavniji algoritam iz klase evolucionih strategija. Smatra se pohlepnim hill climbing ili steepest descent algoritmom.

### Pseudokod algoritma ES(1+1):

1. Inicijalizacija: k ← 0, x_p, σ^k
2. Izračunati fitness y_p ← f(x_p)
3. **repeat**
4.    x_n ← x_p + σ^k * Z^k
5.    y_n ← f(x_n)
6.    **if** y_n < y_p **then**
7.       x_p ← x_n, y_p ← y_n
8.    **end if**
9.    Modificirati σ^k prema pravilu "1/5"
10.   k ← k + 1
11. **until** Ispunjen uslov zaustavljanja

### Pravilo "1/5":

Parametar σ se modificira kao:
- σ^(k+1) = σ^k * a, ako je P_S > 1/5
- σ^(k+1) = σ^k / a, ako je P_S < 1/5
- σ^(k+1) = σ^k, ako je P_S = 1/5

gdje P_S predstavlja uspješnost mutacije, a parametar a se bira između 1.1 i 1.5.

### Zadaci:
1. Implementirati klasu ES za ES(1+1)
2. Implementirati varijantu ES(1,1) - prelazak u novu tačku bez provjere
3. Testirati na funkcijama: Paraboloid, Rastrigin, Drop-Wave, Holder Table

In [1]:
# ============================================================================
# IMPORTOVANJE BIBLIOTEKA
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import random

print("Biblioteke uspjesno ucitane!")

ModuleNotFoundError: No module named 'numpy'

## Testne funkcije

Implementacija testnih funkcija za optimizaciju:
- **Paraboloid** - jednostavna konveksna funkcija
- **Rastrigin** - multimodalna funkcija sa mnogo lokalnih minimuma
- **Drop-Wave** - kompleksna multimodalna funkcija
- **Holder Table** - funkcija sa 4 globalna minimuma

In [ ]:
# ============================================================================
# TESTNE FUNKCIJE ZA OPTIMIZACIJU
# ============================================================================

def paraboloid(x):
    """
    Paraboloid (Sphere) funkcija.
    
    f(x) = sum(x_i^2)
    
    Globalni minimum: f(0, 0) = 0
    Domen: [-5.12, 5.12]^n
    """
    return np.sum(np.array(x)**2)


def rastrigin(x):
    """
    Rastrigin funkcija.
    
    f(x) = 10*n + sum(x_i^2 - 10*cos(2*pi*x_i))
    
    Globalni minimum: f(0, 0) = 0
    Domen: [-5.12, 5.12]^n
    """
    x = np.array(x)
    n = len(x)
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))


def drop_wave(x):
    """
    Drop-Wave funkcija.
    
    f(x) = -( (1 + cos(12*sqrt(x1^2 + x2^2))) / (0.5*(x1^2 + x2^2) + 2) )
    
    Globalni minimum: f(0, 0) = -1
    Domen: [-5.12, 5.12]^2
    """
    x1, x2 = x[0], x[1]
    r_sq = x1**2 + x2**2
    numerator = 1 + np.cos(12 * np.sqrt(r_sq))
    denominator = 0.5 * r_sq + 2
    return -numerator / denominator


def holder_table(x):
    """
    Holder Table funkcija.
    
    f(x) = -|sin(x1)*cos(x2)*exp(|1 - sqrt(x1^2+x2^2)/pi|)|
    
    Globalni minimum: f(+-8.05502, +-9.66459) = -19.2085
    Domen: [-10, 10]^2
    """
    x1, x2 = x[0], x[1]
    term1 = np.sin(x1) * np.cos(x2)
    term2 = np.exp(np.abs(1 - np.sqrt(x1**2 + x2**2) / np.pi))
    return -np.abs(term1 * term2)


# Informacije o funkcijama (za vizualizaciju)
FUNKCIJE = {
    'Paraboloid': {
        'func': paraboloid,
        'bounds': (-5.12, 5.12),
        'global_min': ([0, 0], 0),
        'description': 'f(0, 0) = 0'
    },
    'Rastrigin': {
        'func': rastrigin,
        'bounds': (-5.12, 5.12),
        'global_min': ([0, 0], 0),
        'description': 'f(0, 0) = 0'
    },
    'Drop-Wave': {
        'func': drop_wave,
        'bounds': (-5.12, 5.12),
        'global_min': ([0, 0], -1),
        'description': 'f(0, 0) = -1'
    },
    'Holder Table': {
        'func': holder_table,
        'bounds': (-10, 10),
        'global_min': ([8.05502, 9.66459], -19.2085),
        'description': 'f(+-8.055, +-9.665) = -19.2085'
    }
}

print("Testne funkcije implementirane!")
print("\nDostupne funkcije:")
for name, info in FUNKCIJE.items():
    print(f"  - {name}: {info['description']}")

## Zadatak 1: Implementacija klase ES za ES(1+1)

Klasa ES sa sljedecim atributima i metodama:
- **MaxIter** - maksimalan broj iteracija
- **a** - parametar za pravilo "1/5" (1.1 - 1.5)
- **P_S** - parametar uspjesnosti generisanja boljih jedinki
- **sigma** - trenutna standardna devijacija
- **sigma0** - pocetna matrica standardnih devijacija
- **x** - problemska varijabla
- **eps** - parametar za uslov zaustavljanja
- **mutate()** - metoda za mutaciju na osnovu Gauss-ove raspodjele
- **step()** - jedna iteracija algoritma
- **run()** - pokretanje algoritma

In [ ]:
# ============================================================================
# KLASA ES - Evoluciona strategija (1+1)
# ============================================================================

class ES:
    """
    Implementacija algoritma ES(1+1) - Evoluciona strategija.
    
    Algoritam koristi pravilo "1/5" za adaptaciju parametra sigma.
    Mutacija se vrsi pomocu Gauss-ove raspodjele.
    """
    
    def __init__(self, MaxIter, a, sigma0, fitness_func, x0=None, bounds=None, eps=1e-8, window=10):
        """
        Konstruktor klase ES.
        
        Parametri:
        ----------
        MaxIter : int
            Maksimalan broj iteracija
        a : float
            Parametar za pravilo "1/5" (preporuceno: 1.1 - 1.5)
        sigma0 : float ili np.array
            Pocetna standardna devijacija (dijagonalna matrica)
        fitness_func : callable
            Funkcija cilja koju minimiziramo
        x0 : np.array, optional
            Pocetna tacka (ako None, random inicijalizacija)
        bounds : tuple, optional
            Granice pretrage (min, max)
        eps : float
            Parametar za uslov zaustavljanja
        window : int
            Velicina prozora za racunanje P_S
        """
        self.MaxIter = MaxIter
        self.a = a
        self.sigma0 = sigma0
        self.sigma = np.array(sigma0) if np.isscalar(sigma0) else sigma0.copy()
        self.fitness_func = fitness_func
        self.eps = eps
        self.bounds = bounds
        self.window = window  # Prozor za racunanje P_S
        
        # Inicijalizacija x
        if x0 is not None:
            self.x = np.array(x0, dtype=float)
        elif bounds is not None:
            # Random inicijalizacija unutar granica
            self.x = np.random.uniform(bounds[0], bounds[1], 2)
        else:
            self.x = np.random.uniform(-5, 5, 2)
        
        # Parametar uspjesnosti (P_S)
        self.P_S = 0.0
        self.success_history = []  # Historija uspjesnih mutacija
        
        # Historija za pracenje
        self.x_history = [self.x.copy()]
        self.fitness_history = [self.fitness_func(self.x)]
        self.sigma_history = [np.mean(self.sigma) if hasattr(self.sigma, '__len__') else self.sigma]
    
    def mutate(self):
        """
        Vrsi mutaciju na osnovu Gauss-ove raspodjele.
        
        x_n = x_p + sigma * Z
        
        gdje je Z vektor slucajnih vrijednosti iz N(0, 1).
        
        Vraca:
        ------
        np.array
            Nova mutirana tacka x_n
        """
        # Generisi Z^k - vektor slucajnih vrijednosti iz Gauss-ove raspodjele
        Z = np.array([random.gauss(0, 1) for _ in range(len(self.x))])
        
        # Izracunaj novu tacku
        x_new = self.x + self.sigma * Z
        
        # Primijeni granice ako postoje
        if self.bounds is not None:
            x_new = np.clip(x_new, self.bounds[0], self.bounds[1])
        
        return x_new
    
    def update_sigma(self):
        """
        Modificira sigma prema pravilu "1/5".
        
        - Ako P_S > 1/5: sigma = sigma * a (povecaj - previse uspjesnih mutacija)
        - Ako P_S < 1/5: sigma = sigma / a (smanji - premalo uspjesnih mutacija)
        - Ako P_S = 1/5: sigma ostaje isti
        """
        if self.P_S > 0.2:  # 1/5 = 0.2
            self.sigma = self.sigma * self.a
        elif self.P_S < 0.2:
            self.sigma = self.sigma / self.a
        # Ako je P_S == 0.2, sigma ostaje isti
    
    def step(self):
        """
        Izvrsava jednu iteraciju algoritma ES(1+1).
        
        1. Generisi novu tacku mutacijom
        2. Evaluiraj fitness
        3. Ako je bolja, prihvati
        4. Azuriraj P_S i sigma
        
        Vraca:
        ------
        bool
            True ako je mutacija bila uspjesna
        """
        # Trenutni fitness
        y_p = self.fitness_func(self.x)
        
        # Generisi novu tacku
        x_n = self.mutate()
        y_n = self.fitness_func(x_n)
        
        # Provjeri da li je nova tacka bolja (minimizacija)
        success = False
        if y_n < y_p:
            self.x = x_n
            success = True
        
        # Azuriraj historiju uspjesnosti
        self.success_history.append(1 if success else 0)
        if len(self.success_history) > self.window:
            self.success_history.pop(0)
        
        # Izracunaj P_S (uspjesnost u zadnjih 'window' iteracija)
        self.P_S = sum(self.success_history) / len(self.success_history)
        
        # Azuriraj sigma prema pravilu 1/5
        self.update_sigma()
        
        # Snimi historiju
        self.x_history.append(self.x.copy())
        self.fitness_history.append(self.fitness_func(self.x))
        self.sigma_history.append(np.mean(self.sigma) if hasattr(self.sigma, '__len__') else self.sigma)
        
        return success
    
    def run(self, verbose=True):
        """
        Pokrece izvrsavanje algoritma ES(1+1).
        
        Parametri:
        ----------
        verbose : bool
            Ako True, ispisuje napredak
        
        Vraca:
        ------
        tuple (np.array, float)
            Najbolja pronadjena tacka i njena vrijednost
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"ES(1+1) ALGORITAM")
            print(f"{'='*60}")
            print(f"Max iteracija: {self.MaxIter}")
            print(f"Parametar a: {self.a}")
            print(f"Pocetni sigma: {self.sigma0}")
            print(f"Pocetna tacka: {self.x}")
            print(f"Pocetni fitness: {self.fitness_history[0]:.6f}")
            print(f"{'='*60}\n")
        
        for k in range(self.MaxIter):
            # Izvrsi jedan korak
            self.step()
            
            # Uslov zaustavljanja - ako je sigma premali
            if np.all(self.sigma < self.eps):
                if verbose:
                    print(f"\nZaustavljanje: sigma < eps na iteraciji {k+1}")
                break
            
            # Ispis napretka
            if verbose and (k % 100 == 0 or k == self.MaxIter - 1):
                print(f"Iter {k+1:4d}: x = [{self.x[0]:8.5f}, {self.x[1]:8.5f}], "
                      f"f(x) = {self.fitness_history[-1]:10.6f}, "
                      f"sigma = {np.mean(self.sigma):.6f}, P_S = {self.P_S:.3f}")
        
        # Rezultat
        best_idx = np.argmin(self.fitness_history)
        best_x = self.x_history[best_idx]
        best_fitness = self.fitness_history[best_idx]
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"REZULTAT:")
            print(f"  Najbolja tacka: [{best_x[0]:.6f}, {best_x[1]:.6f}]")
            print(f"  Najbolji fitness: {best_fitness:.6f}")
            print(f"{'='*60}")
        
        return best_x, best_fitness


print("Klasa ES (1+1) implementirana!")

## Zadatak 2: Implementacija ES(1,1) varijante

ES(1,1) varijanta prelazi u novu tacku x_n bez obzira je li ona bolja ili losija od trenutne tacke x_p. Ne vrsi se provjera kao u liniji 6 originalnog pseudokoda.

In [ ]:
# ============================================================================
# KLASA ES_Comma - Evoluciona strategija (1,1)
# ============================================================================

class ES_Comma(ES):
    """
    Implementacija algoritma ES(1,1) - varijanta bez selekcije.
    
    U ovoj varijanti se UVIJEK prelazi u novu tacku,
    bez obzira je li ona bolja ili losija od trenutne.
    """
    
    def __init__(self, MaxIter, a, sigma0, fitness_func, x0=None, bounds=None, eps=1e-8, window=10):
        """
        Konstruktor - nasljeduje sve od ES klase.
        """
        super().__init__(MaxIter, a, sigma0, fitness_func, x0, bounds, eps, window)
        
        # Dodatno: pratimo najbolju pronadjenu tacku
        self.best_x = self.x.copy()
        self.best_fitness = self.fitness_func(self.x)
    
    def step(self):
        """
        Izvrsava jednu iteraciju algoritma ES(1,1).
        
        RAZLIKA od ES(1+1): Uvijek prelazimo u novu tacku!
        
        1. Generisi novu tacku mutacijom
        2. Evaluiraj fitness
        3. UVIJEK prihvati novu tacku
        4. Azuriraj P_S i sigma
        5. Zapamti ako je najbolja
        
        Vraca:
        ------
        bool
            True ako je nova tacka bolja (za statistiku)
        """
        # Trenutni fitness
        y_p = self.fitness_func(self.x)
        
        # Generisi novu tacku
        x_n = self.mutate()
        y_n = self.fitness_func(x_n)
        
        # Provjeri da li je nova tacka bolja (za P_S)
        success = y_n < y_p
        
        # ES(1,1): UVIJEK prihvati novu tacku!
        self.x = x_n
        
        # Azuriraj najbolju tacku ako je potrebno
        if y_n < self.best_fitness:
            self.best_x = x_n.copy()
            self.best_fitness = y_n
        
        # Azuriraj historiju uspjesnosti
        self.success_history.append(1 if success else 0)
        if len(self.success_history) > self.window:
            self.success_history.pop(0)
        
        # Izracunaj P_S
        self.P_S = sum(self.success_history) / len(self.success_history)
        
        # Azuriraj sigma prema pravilu 1/5
        self.update_sigma()
        
        # Snimi historiju
        self.x_history.append(self.x.copy())
        self.fitness_history.append(y_n)
        self.sigma_history.append(np.mean(self.sigma) if hasattr(self.sigma, '__len__') else self.sigma)
        
        return success
    
    def run(self, verbose=True):
        """
        Pokrece izvrsavanje algoritma ES(1,1).
        
        Vraca najbolju pronadjenu tacku (ne nuzno trenutnu).
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"ES(1,1) ALGORITAM (bez selekcije)")
            print(f"{'='*60}")
            print(f"Max iteracija: {self.MaxIter}")
            print(f"Parametar a: {self.a}")
            print(f"Pocetni sigma: {self.sigma0}")
            print(f"Pocetna tacka: {self.x}")
            print(f"Pocetni fitness: {self.fitness_history[0]:.6f}")
            print(f"{'='*60}\n")
        
        for k in range(self.MaxIter):
            self.step()
            
            # Uslov zaustavljanja
            if np.all(self.sigma < self.eps):
                if verbose:
                    print(f"\nZaustavljanje: sigma < eps na iteraciji {k+1}")
                break
            
            if verbose and (k % 100 == 0 or k == self.MaxIter - 1):
                print(f"Iter {k+1:4d}: x = [{self.x[0]:8.5f}, {self.x[1]:8.5f}], "
                      f"f(x) = {self.fitness_func(self.x):10.6f}, "
                      f"best = {self.best_fitness:10.6f}, "
                      f"sigma = {np.mean(self.sigma):.6f}")
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"REZULTAT:")
            print(f"  Najbolja tacka: [{self.best_x[0]:.6f}, {self.best_x[1]:.6f}]")
            print(f"  Najbolji fitness: {self.best_fitness:.6f}")
            print(f"{'='*60}")
        
        return self.best_x, self.best_fitness


print("Klasa ES_Comma (1,1) implementirana!")

## Zadatak 3: Testiranje i vizualizacija

Testiranje obje varijante algoritma na funkcijama:
- Paraboloid
- Rastrigin  
- Drop-Wave
- Holder Table

Svaka funkcija ce biti prikazana graficki sa:
- Crno-bijeli contour plot
- Crveni kruzic (o) za globalni minimum
- Krizic (x) za pronadjenu tacku

In [ ]:
# ============================================================================
# FUNKCIJA ZA VIZUALIZACIJU
# ============================================================================

def plot_results(func_name, es_plus_result, es_comma_result, func_info):
    """
    Vizualizira rezultate za jednu funkciju.
    
    Parametri:
    ----------
    func_name : str
        Naziv funkcije
    es_plus_result : tuple
        Rezultat ES(1+1): (best_x, best_fitness)
    es_comma_result : tuple
        Rezultat ES(1,1): (best_x, best_fitness)
    func_info : dict
        Informacije o funkciji
    """
    func = func_info['func']
    bounds = func_info['bounds']
    global_min_x, global_min_val = func_info['global_min']
    
    # Kreiraj mrezu tacaka
    x_range = np.linspace(bounds[0], bounds[1], 200)
    y_range = np.linspace(bounds[0], bounds[1], 200)
    X, Y = np.meshgrid(x_range, y_range)
    
    # Izracunaj vrijednosti funkcije
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = func([X[i, j], Y[i, j]])
    
    # Kreiraj figure sa 2 subplota
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for idx, (ax, result, title) in enumerate([
        (axes[0], es_plus_result, 'ES(1+1)'),
        (axes[1], es_comma_result, 'ES(1,1)')
    ]):
        # Crno-bijeli contour plot
        contour = ax.contour(X, Y, Z, levels=20, colors='black', linewidths=0.5)
        ax.contourf(X, Y, Z, levels=20, cmap='Greys')
        
        # Globalni minimum - crveni kruzic
        ax.plot(global_min_x[0], global_min_x[1], 'ro', markersize=12, 
                label=f'Globalni min ({global_min_x[0]:.2f}, {global_min_x[1]:.2f})', markeredgecolor='darkred', markeredgewidth=2)
        
        # Pronadjena tacka - krizic
        best_x = result[0]
        ax.plot(best_x[0], best_x[1], 'bx', markersize=12, markeredgewidth=3,
                label=f'Pronadjeno ({best_x[0]:.4f}, {best_x[1]:.4f})')
        
        # Postavke
        ax.set_xlabel('x1', fontsize=12)
        ax.set_ylabel('x2', fontsize=12)
        ax.set_title(f'{func_name} - {title}\nf(x) = {result[1]:.6f}', fontsize=14, fontweight='bold')
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(bounds[0], bounds[1])
        ax.set_ylim(bounds[0], bounds[1])
    
    plt.tight_layout()
    plt.show()


print("Funkcija za vizualizaciju implementirana!")

In [ ]:
# ============================================================================
# TESTIRANJE NA SVIM FUNKCIJAMA
# ============================================================================

# Parametri algoritma
MAX_ITER = 500
A = 1.3  # Parametar za pravilo 1/5
SIGMA0 = 1.0  # Pocetna standardna devijacija

# Postavi seed za reproducibilnost
np.random.seed(42)
random.seed(42)

print("="*70)
print("TESTIRANJE ALGORITAMA ES(1+1) i ES(1,1)")
print("="*70)
print(f"Parametri: MaxIter={MAX_ITER}, a={A}, sigma0={SIGMA0}")
print("="*70)

# Rezultati za sve funkcije
results = {}

for func_name, func_info in FUNKCIJE.items():
    print(f"\n\n{'#'*70}")
    print(f"# FUNKCIJA: {func_name}")
    print(f"# Globalni minimum: {func_info['description']}")
    print(f"{'#'*70}")
    
    # Ista pocetna tacka za fer poredenje
    x0 = np.random.uniform(func_info['bounds'][0], func_info['bounds'][1], 2)
    
    # ES(1+1)
    print("\n--- ES(1+1) ---")
    es_plus = ES(
        MaxIter=MAX_ITER,
        a=A,
        sigma0=SIGMA0,
        fitness_func=func_info['func'],
        x0=x0.copy(),
        bounds=func_info['bounds']
    )
    result_plus = es_plus.run(verbose=True)
    
    # ES(1,1)
    print("\n--- ES(1,1) ---")
    es_comma = ES_Comma(
        MaxIter=MAX_ITER,
        a=A,
        sigma0=SIGMA0,
        fitness_func=func_info['func'],
        x0=x0.copy(),
        bounds=func_info['bounds']
    )
    result_comma = es_comma.run(verbose=True)
    
    # Sacuvaj rezultate
    results[func_name] = {
        'es_plus': result_plus,
        'es_comma': result_comma,
        'es_plus_obj': es_plus,
        'es_comma_obj': es_comma
    }
    
    # Vizualizacija
    plot_results(func_name, result_plus, result_comma, func_info)

In [ ]:
# ============================================================================
# SUMARNI PREGLED REZULTATA
# ============================================================================

print("\n" + "="*90)
print("SUMARNI PREGLED REZULTATA")
print("="*90)
print(f"{'Funkcija':<15} | {'ES(1+1)':<35} | {'ES(1,1)':<35}")
print("-"*90)

for func_name, res in results.items():
    global_min = FUNKCIJE[func_name]['global_min'][1]
    
    plus_x, plus_f = res['es_plus']
    comma_x, comma_f = res['es_comma']
    
    plus_error = abs(plus_f - global_min)
    comma_error = abs(comma_f - global_min)
    
    plus_str = f"f={plus_f:10.6f} (err={plus_error:.6f})"
    comma_str = f"f={comma_f:10.6f} (err={comma_error:.6f})"
    
    # Oznaci bolji rezultat
    if plus_error < comma_error:
        plus_str += " *"
    elif comma_error < plus_error:
        comma_str += " *"
    
    print(f"{func_name:<15} | {plus_str:<35} | {comma_str:<35}")

print("-"*90)
print("* oznacava bolji rezultat")
print("="*90)

In [ ]:
# ============================================================================
# PRIKAZ KONVERGENCIJE
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (func_name, res) in enumerate(results.items()):
    ax = axes[idx]
    
    es_plus = res['es_plus_obj']
    es_comma = res['es_comma_obj']
    
    # Plot fitness historije
    ax.plot(es_plus.fitness_history, 'b-', linewidth=1.5, label='ES(1+1)', alpha=0.8)
    ax.plot(es_comma.fitness_history, 'r--', linewidth=1.5, label='ES(1,1)', alpha=0.8)
    
    # Globalni minimum
    global_min = FUNKCIJE[func_name]['global_min'][1]
    ax.axhline(y=global_min, color='g', linestyle=':', linewidth=2, label=f'Globalni min ({global_min})')
    
    ax.set_xlabel('Iteracija', fontsize=11)
    ax.set_ylabel('Fitness', fontsize=11)
    ax.set_title(f'{func_name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('symlog')  # Simetricna log skala za bolje prikazivanje

plt.suptitle('Konvergencija algoritama', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Zakljucak

### Implementirane komponente:

1. **ES(1+1) algoritam** - Evoluciona strategija sa selekcijom
   - Prihvata novu tacku samo ako je bolja od trenutne
   - Koristi pravilo "1/5" za adaptaciju sigma
   - Garantuje monotono opadanje fitness vrijednosti

2. **ES(1,1) algoritam** - Evoluciona strategija bez selekcije
   - Uvijek prelazi u novu tacku
   - Pamti najbolju pronadjenu tacku
   - Moze "pobijeci" iz lokalnih minimuma

3. **Testne funkcije:**
   - Paraboloid - jednostavna konveksna funkcija
   - Rastrigin - multimodalna sa mnogo lokalnih minimuma
   - Drop-Wave - kompleksna multimodalna funkcija
   - Holder Table - funkcija sa 4 globalna minimuma

### Analiza rezultata:

**ES(1+1) prednosti:**
- Stabilnija konvergencija
- Garantovano ne pogorsava trenutno rjesenje
- Bolje performanse na jednostavnim, konveksnim funkcijama (Paraboloid)

**ES(1,1) prednosti:**
- Veca eksloracija prostora pretrage
- Moze izbjeci lokalne minimume
- Potencijalno bolje na multimodalnim funkcijama

**Zakljucak:**
ES(1+1) generalno daje bolje i stabilnije rezultate jer zadrzava najbolje pronadjeno rjesenje. ES(1,1) ima vecu tendenciju eksploratije sto moze biti korisno za izbjegavanje lokalnih minimuma, ali na cijelinu daje losije rezultate zbog nestabilnosti.